# LUNES 

<b> probabilidad | ¿Qué nos dice la distribución empírica sobre el riesgo?
Modalidad: Código. </b>

Imagina que tenemos tres clases de activos: acciones, bonos y oro.

Primero construiremos 252 observaciones sintéticas de retornos diarios.

```python
import numpy as np
import pandas as pd

rng = np.random.default_rng(117)

T = 252
z = rng.standard_normal((T, 3))

equity = 0.00045 + 0.011 * z[:, 0]

bonds = (
    0.00015
    + 0.003 * (
        0.2 * z[:, 0]
        + np.sqrt(0.96) * z[:, 1]
    )
)

gold = (
    0.00020
    + 0.008 * (
```

Todos los retornos están expresados en formato decimal.

No son observaciones reales de mercado.

**Parte A — Construye dos portafolios**

Portfolio A:

$$w_A = \begin{bmatrix} 0.60 \\ 0.25 \\ 0.15 \end{bmatrix}$$


Portfolio B:

$$w_B = \begin{bmatrix} 0.40 \\ 0.35 \\ 0.25 \end{bmatrix}.$$

Supondremos pesos constantes mediante rebalanceo diario idealizado, sin costes.

Calcula:

$$R_{P,t} = w^\top R_t.$$

Para cada portafolio estima:

$$\hat{\mu} = \frac{1}{T} \sum_t R_{P,t}$$

$$\hat{\sigma} = \text{Std}(R_P).$$

Anualiza:

$$\mu_{\text{ann}} = 252\hat{\mu}$$

$$\sigma_{\text{ann}} = \sqrt{252}\hat{\sigma}.$$

La media anualizada es una aproximación aritmética, no el rendimiento compuesto anual realizado.



**Parte B — Riesgo de cola**

Obtén el cuantil empírico del 5%:

$$q_{0.05}(R_P).$$

Define el VaR histórico al 95%, expresado como pérdida positiva:

$$VaR_{95\%} = -q_{0.05}(R_P).$$

Después estima:

$$ES_{95\%} = -E[R_P \mid R_P \leq q_{0.05}].$$

Finalmente calcula el máximo drawdown a partir de:

$$W_t = \prod_{s=1}^t (1 + R_{P,s}),$$

incluyendo $W_0 = 1$.

Construye esta tabla:

| Métrica | Portfolio A | Portfolio B |
|---|---:|---:|
| Annualized Mean Return | | |
| Annualized Volatility | | |
| Historical VaR 95% | | |
| Historical ES 95% | | |
| Maximum Drawdown | | |

**Parte C — Interpretación**

Responde tres preguntas.

1. ¿Reducir volatilidad implica necesariamente reducir Expected Shortfall?

2. ¿Por qué un evento de estrés puede afectar el drawdown de manera distinta a la volatilidad muestral?

3. Si el Portfolio B presenta mejores métricas históricas de riesgo, ¿puedes concluir que será superior fuera de muestra?

Considera que disponemos de una muestra relativamente pequeña para estudiar eventos extremos y que los parámetros de la simulación son conocidos.

**Entregable:** Un DataFrame comparativo y tres conclusiones.

No optimices los pesos. Los dos portafolios fueron definidos antes de evaluar sus resultados.



In [6]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(117)

T = 252
z = rng.standard_normal((T, 3))

equity = 0.00045 + 0.011 * z[:, 0]

bonds = (
    0.00015
    + 0.003 * (
        0.2 * z[:, 0]
        + np.sqrt(0.96) * z[:, 1]
    )
)

gold = (
    0.00020
    + 0.008 * (
        0.1 * z[:, 0]
        + np.sqrt(0.99) * z[:, 2]
    )
)

R = np.column_stack([equity, bonds, gold])

# Escenario de estrés sintético
R[180] = [-0.06, -0.012, 0.016]

# Construccion de los 2 portafolios 
w_A = np.array([0.60, 0.25, 0.15])
w_B = np.array([0.40, 0.35, 0.25])
# supondremnos pesos constantes mediante rebalanaceo diario, sin costes 
def portfolio_calculos(R, w):
    # Retorno diario del portfolio: 252 observaciones
    portfolio_returns = R @ w

    # Parte A
    mu_daily = portfolio_returns.mean()
    sigma_daily = portfolio_returns.std(ddof=1)

    mu_ann = 252 * mu_daily
    sigma_ann = np.sqrt(252) * sigma_daily

    # Parte B: riesgo de cola
    q_05 = np.quantile(portfolio_returns, 0.05)

    # Pérdida positiva
    var_95 = -q_05

    tail_returns = portfolio_returns[portfolio_returns <= q_05]
    es_95 = -tail_returns.mean()

    # Wealth con W_0 = 1
    wealth = np.r_[1.0, np.cumprod(1 + portfolio_returns)]
    peak = np.maximum.accumulate(wealth)
    drawdown = wealth / peak - 1
    max_drawdown = drawdown.min()

    return {
        "Annualized Mean Return": mu_ann,
        "Annualized Volatility": sigma_ann,
        "Historical VaR 95%": var_95,
        "Historical ES 95%": es_95,
        "Maximum Drawdown": max_drawdown
    }

metricas_A = portfolio_calculos(R, w_A)
metricas_B = portfolio_calculos(R, w_B)

df_metricas = pd.DataFrame({
    "Portfolio A": metricas_A,
    "Portfolio B": metricas_B
})

df_metricas

,Portfolio A,Portfolio B
Annualized Mean Return,0.182980,0.147736
Annualized Volatility,0.116894,0.084665
Historical VaR 95%,0.010863,0.008082
Historical ES 95%,0.015766,0.011493
Maximum Drawdown,-0.059917,-0.047482


1. ¿Reducir volatilidad implica necesariamente reducir Expected Shortfall?
- R: 
> el Expected Shortfall mide exclusivamente la pérdida media dentro del peor 5% de los retornos.
Un portfolio puede tener menor volatilidad promedio, pero seguir expuesto a pérdidas extremas por asimetría negativa, colas pesadas, saltos de precio o correlaciones que aumentan durante estrés. En ese caso, su ES puede ser igual o incluso mayor que el de un portfolio más volátil.

2. ¿Por qué un evento de estrés puede afectar el drawdown de manera distinta a la volatilidad muestral?

3. Si el Portfolio B presenta mejores métricas históricas de riesgo, ¿puedes concluir que será superior fuera de muestra?

Considera que disponemos de una muestra relativamente pequeña para estudiar eventos extremos y que los parámetros de la simulación son conocidos.

# Martes

**Estadística | detectar y reparar una matriz indefinida**

Supongamos que recibes esta matriz de correlaciones calculada mediante estimaciones por pares de activos:

$$C = \begin{bmatrix} 1 & 0.90 & -0.90 \\ 0.90 & 1 & 0.90 \\ -0.90 & 0.90 & 1 \end{bmatrix}$$

Cada correlación individual parece encontrarse dentro del rango permitido.

Sin embargo, eso no garantiza que la matriz completa sea válida.

Utiliza:

```python
import numpy as np

C = np.array([
    [1.00, 0.90, -0.90],
    [0.90, 1.00, 0.90],
    [-0.90, 0.90, 1.00]
])

vol = np.array([0.01, 0.015, 0.012])

Sigma = np.outer(vol, vol) * C
```

**Parte A — Validación espectral (7 minutos)**

Calcula los eigenvalues de \[C\]:

```python
eigenvalues, eigenvectors = np.linalg.eigh(C)
```

Comprueba:

$$\lambda_{\min}(C) \geq 0$$

Después verifica las tres propiedades:

$$C = C^T$$

$$C_{ii} = 1$$

$$C \succeq 0$$

**Pregunta: ¿Por qué verificar únicamente que todas las correlaciones estén entre -1 y 1 resulta insuficiente?**

**Parte B — Reparación numérica (8 minutos)**

Implementa una reparación sencilla mediante eigenvalue clipping:

$$\lambda_i^* = \max(\lambda_i, \epsilon)$$

con:

$$\epsilon = 10^{-8}$$

Reconstruye:
$$\tilde{C} = Q \Lambda^* Q^T$$

Después normaliza la matriz reconstruida para que su diagonal vuelva a contener unos:

$$C_{ij}^{repaired} = \frac{\tilde{C}_{ij}}{\sqrt{\tilde{C}_{ii} \tilde{C}_{jj}}}$$

Verifica nuevamente su simetría, diagonal y eigenvalue mínimo, admitiendo tolerancia numérica.

Esta es una reparación ilustrativa; no equivale necesariamente a encontrar la matriz de correlación válida más cercana ni demuestra que las dependencias corregidas sean las verdaderas.

**Parte C — Evaluación (5 minutos)**

Calcula:

$$\|C^{repaired} - C\|_F$$

La norma de Frobenius cuantifica cuánto modificaste la matriz.

Responde:

> ¿Por qué reparar numéricamente una matriz no demuestra que el problema de calidad de datos haya sido solucionado?

> Una matriz de correlaciones inválida (no definida positiva) suele originarse por datos inconsistentes, como calcular coeficientes utilizando ventanas de tiempo distintas para cada par de activos, la presencia de valores atípicos (outliers) o la falta de observaciones alineadas cronológicamente. El algoritmo matemático simplemente modifica los valores para cumplir de manera artificial con las restricciones geométricas $ C \succeq 0 $ , pero no recupera la verdadera relación económica ni la consistencia empírica de los datos históricos subyacentes.


**Entregable:** Una tabla con las correlaciones originales y reparadas, sus eigenvalues mínimos y la magnitud de la modificación.

No necesitas crear un optimizador ni incorporar este procedimiento a tus repositorios.


In [ ]:
import numpy as np

C = np.array([
    [1.00, 0.90, -0.90],
    [0.90, 1.00, 0.90],
    [-0.90, 0.90, 1.00]
])

vol = np.array([0.01, 0.015, 0.012])

Sigma = np.outer(vol, vol) * C

<B> Market Liquidity, Trading Capacity & Execution Risk </B>

- Tema: ¿Cuánto capital puede absorber una estrategia antes de perder su rentabilidad?

Continuamos con el bloque semanal de Portfolio Risk y mercados financieros. Después de estudiar la estimación y validez de matrices de covarianza, hoy abordaremos un problema diferente: una estrategia puede presentar alpha positivo y, aun así, no ser ejecutable al tamaño deseado.

La pregunta central será:

$$\text{Statistical Alpha} \neq \text{Executable Alpha}$$

Hoy no necesitamos consultar cotizaciones reales ni modificar tus repositorios. Trabajaremos con un caso sintético que permita comprender las relaciones matemáticas antes de utilizar datos de Alpha Vantage.


| Ejercicio | Competencia | Modalidad | Tiempo |
|---|---|---|---:|
| 1 | Matemáticas financieras y costes | A mano | 20 min |
| 2 | Estadística de liquidez y capacidad | Python | 20 min |
| 3 | Research, mercados y Quant Desk | Lectura + razonamiento | 20 min |
| **Total** | | | **60 min** |

```python
import numpy as np
import pandas as pd

# Volúmenes en 12 intervalos de 5 minutos
volume = np.array([
    1200, 1000, 800, 0, 900, 1100,
    300, 100, 0, 200, np.nan, 100
])

# Regímenes: 6 normales y 6 de estrés (menor actividad)
regime = np.array(
    ["NORMAL"] * 6 + ["STRESS"] * 6
)

# Creación del DataFrame
df = pd.DataFrame({
    "volume": volume,
    "regime": regime
})

# Parámetros de ejecución
target_quantity = 200
max_participation = 0.10
```

La política de ejecución establece que nuestra participación no debe superar el 10% del volumen negociado en cada intervalo.

Para una barra válida:

$$\text{Capacity}_t = 0.10 \times \text{Volume}_t$$

El máximo volumen observable que podríamos intentar ejecutar bajo esa restricción es:

$$Q_{\max} = \sum_{t} \text{Capacity}_t$$

Este cálculo es únicamente una aproximación de capacidad basada en volumen. No garantiza que nuestras órdenes encuentren contraparte o se ejecuten al precio deseado.

### Parte A — Estimación de capacidad (8 minutos)

Calcula para cada régimen:

$$Q_{\max, \text{NORMAL}}$$

$$Q_{\max, \text{STRESS}}$$

Construye esta tabla:

| Métrica | Normal | Stress |
|---|---:|---:|
| Volumen observado | | |
| Barras válidas | | |
| Capacidad observada al 10% | | |
| Orden objetivo | 200 | 200 |
| ¿Capacidad suficiente en barras observadas? | | |

¿Capacidad suficiente en barras observadas?

**Atención a los datos:** Un volumen cero válido significa que no hubo operaciones registradas. Un valor `NaN` significa que desconocemos el volumen; no debes sustituirlo por cero para afirmar que observamos ausencia de operaciones.

### Parte B — Problema de ejecución (6 minutos)

Supongamos que la estrategia necesita completar una orden de 200 títulos durante 30 minutos.

Responde:

1. ¿Qué régimen proporciona mayor capacidad observable?
2. ¿Qué sucede si el volumen disponible disminuye considerablemente?
3. ¿Es correcto concluir que una orden no puede completarse durante todo el periodo de estrés cuando una barra carece de información?

La tercera pregunta requiere diferenciar capacidad observada de capacidad real desconocida.

### Parte C — Statistical Research (6 minutos)

Ahora imagina que calculaste esta capacidad durante 100 ventanas históricas.

Obtienes:

$$P(Q_{\max} \geq 200) = 0.72$$

Research concluye:

> Nuestra estrategia tiene una probabilidad del 72% de completar sus órdenes.

¿Es estadísticamente correcta esa afirmación?

Explica por qué la probabilidad empírica de observar volumen suficiente no equivale necesariamente a una probabilidad de ejecución.

Considera prioridad en la cola, órdenes competidoras, movimiento de precios y selección temporal de las ventanas.

**Entregable:** Un DataFrame comparativo y una conclusión de máximo cinco líneas.

No necesitas implementar un simulador de order book.
